#### Based on: https://stackoverflow.com/questions/60663073/how-can-i-properly-run-detectron2-on-videos

In [ ]:
import cv2
import torch
import numpy as np
import mss
import detectron2
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.utils.video_visualizer import VideoVisualizer
from detectron2.data import MetadataCatalog
import os

In [ ]:
basedir = "your/path/"

cfg = get_cfg()
cfg.merge_from_file(os.path.join(basedir, "detectron2Files", "mask_rcnn_R_50_FPN_3x.yaml"))

cfg.MODEL.DEVICE = 'cpu'
cfg.DATALOADER.NUM_WORKERS = 4
cfg.SOLVER.IMS_PER_BATCH = 4  # This is the real "batch size" commonly known to deep learning people
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # The "RoIHead batch size". 128 is faster, and good enough for this toy dataset (default: 512)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.WEIGHTS = os.path.join(basedir, 'models', 'model_final_resnet50.pth')  # path to the model we just trained
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.9   # set a custom testing threshold
predictor = DefaultPredictor(cfg)

In [ ]:
# Extract video properties
video = cv2.VideoCapture('/path/to/video.mp4')
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
frames_per_second = video.get(cv2.CAP_PROP_FPS)
num_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

In [ ]:
num_frames

In [ ]:
# Initialize video writer
video_writer = cv2.VideoWriter('output.mp4', fourcc=cv2.VideoWriter_fourcc(*"mp4v"), fps=float(frames_per_second), frameSize=(width, height), isColor=True)

In [ ]:
# Initialize visualizer
v = VideoVisualizer(MetadataCatalog.get(cfg.DATASETS.TRAIN[0]))

In [ ]:
from utils import Utils

def runOnVideo(video, maxFrames):
    """ Runs the predictor on every frame in the video (unless maxFrames is given),
    and returns the frame with the predictions drawn.
    """

    readFrames = 0
    while True:
        hasFrame, frame = video.read()
        if not hasFrame:
            break
        #width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
        #height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

        #resize = (width//4, height//4)
        # Resize the frame to the desired size
        #frame = cv2.resize(frame, resize, interpolation=cv2.INTER_LINEAR)
        # Get prediction results for this frame
        outputs = predictor(frame)


        # Make sure the frame is colored
        #frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)


        # # Draw a visualization of the predictions using the video visualizer on the person class
        # visualization = v.draw_instance_predictions(frame, outputs["instances"][outputs['instances'].pred_classes == 0].to("cpu"))
        predictions = outputs["instances"].to("cpu")
    
        masks = predictions.pred_masks if predictions.has("pred_masks") else None

        for mask in masks:
                        
            # Find mask contour using skimage find_contours
            contours, _  = Utils().get_mask_contours(mask)

            # If more than one contour found in mask, get the longest
            list_len = [len(i) for i in contours]
            longest_index = np.argmax(np.array(list_len))
            # Sometimes two polygons are found in the same detection, we want to take the last - which is the biggest
            paired_list = list(zip(contours[longest_index][::2], contours[longest_index][1::2]))
            # Convert tuples to lists (optional)
            contours = [list(pair) for pair in paired_list]

            points = np.array(contours)

            x = points[:, 0]  # Extract x-coordinates
            y = points[:, 1]  # Extract y-coordinates
            draw_polygon = np.column_stack((x, y)).astype(np.int32)   
            # Draw the bumpy circle contour
            img = cv2.polylines(frame, [draw_polygon], isClosed=True, color=(0, 0, 255), thickness=5)
        
        # Resize the image to the original video size
        #img = cv2.resize(img, (width, height), interpolation=cv2.INTER_LINEAR)

        # Convert Matplotlib RGB format to OpenCV BGR format
        #visualization = cv2.cvtColor(visualization.get_image(), cv2.COLOR_RGB2BGR)

        yield img

        readFrames += 1
        if readFrames > maxFrames:
            break

In [ ]:
import tqdm

# Create a cut-off for debugging
num_frames = 112

# Enumerate the frames of the video
for visualization in tqdm.tqdm(runOnVideo(video, num_frames), total=num_frames):

    # Write to video file
    video_writer.write(visualization)

# Release resources
video.release()
video_writer.release()
#cv2.destroyAllWindows()